# Hooks

> Claude Code hook scripts: safecmd (PreToolUse), exhash (PostToolUse), and code indexer (SessionStart)

## safecmd PreToolUse Hook

Intercepts every Bash tool call. Reads the command from stdin JSON, validates it against safecmd's allowlist, and either allows it (exit 0) or blocks it (returns deny decision via stdout JSON).

In [ ]:
#| default_exp hooks.safecmd_hook

In [ ]:
#| export
from __future__ import annotations
import json
import sys
from pathlib import Path

In [ ]:
#| export
# Default allowlist for common dev commands.
# The setup command writes this to .claude/safecmd_allowlist.json
DEFAULT_ALLOWLIST = [
    # Package management
    "uv", "pip", "pip3", "pipx",
    # Python
    "python", "python3", "ipython",
    # nbdev
    "nbdev-export", "nbdev-test", "nbdev-docs", "nbdev-prepare",
    "nbdev-clean", "nbdev-install-hooks", "nbdev-new",
    # Git (read-only and safe write ops)
    "git",
    # Testing
    "pytest", "coverage",
    # Jupyter
    "jupyter", "nbconvert",
    # File ops (safe subset)
    "ls", "cat", "head", "tail", "grep", "find", "wc", "sort", "uniq",
    "mkdir", "touch", "cp", "mv",
    # Claude Code
    "claude",
    # Build tools
    "make", "cargo",
    # Text processing
    "echo", "printf", "sed", "awk", "cut", "tr",
    # Network (read-only)
    "curl", "wget",
]

In [ ]:
#| export
def load_allowlist(claude_dir: Path | None = None) -> list[str]:
    """Load the safecmd allowlist from .claude/safecmd_allowlist.json, falling back to DEFAULT_ALLOWLIST."""
    if claude_dir is None:
        claude_dir = Path.cwd() / '.claude'
    allowlist_path = claude_dir / 'safecmd_allowlist.json'
    if allowlist_path.exists():
        data = json.loads(allowlist_path.read_text())
        return data.get('allowed_commands', DEFAULT_ALLOWLIST)
    return DEFAULT_ALLOWLIST

In [ ]:
#| export
def validate_command(command: str, allowlist: list[str]) -> tuple[bool, str]:
    """Check if `command` is permitted. Returns (allowed, reason).
    
    First tries safecmd library if available; falls back to prefix-matching against allowlist.
    """
    # First try safecmd library for AST-based validation
    try:
        from safecmd import check  # safecmd public API
        result = check(command, allowed=allowlist)
        if result.allowed:
            return True, 'allowed by safecmd'
        else:
            return False, f'blocked by safecmd: {result.reason}'
    except ImportError:
        pass  # fallback below
    except Exception:
        pass  # on any safecmd error, fall through to simple check

    # Fallback: simple first-token check
    stripped = command.strip()
    if not stripped:
        return True, 'empty command'

    # Block obviously destructive patterns regardless of allowlist
    BLOCKED_PATTERNS = ['rm -rf /', 'rm -rf ~', ':(){ :|:& };:', '> /dev/', 'dd if=', 'mkfs']
    for pat in BLOCKED_PATTERNS:
        if pat in stripped:
            return False, f'blocked: contains dangerous pattern "{pat}"'

    first_token = stripped.split()[0]
    # Strip path prefix (e.g. /usr/bin/python -> python)
    binary = Path(first_token).name

    if binary in allowlist:
        return True, f'allowed: {binary} is in allowlist'

    return False, f'blocked: "{binary}" is not in the allowed commands list'

In [ ]:
#| export
def main():
    """Claude Code PreToolUse hook entry point. Reads JSON from stdin, writes decision to stdout."""
    try:
        payload = json.load(sys.stdin)
    except (json.JSONDecodeError, EOFError):
        sys.exit(0)  # No input or bad JSON: don't block

    tool_name = payload.get('tool_name', '')
    if tool_name != 'Bash':
        sys.exit(0)  # Not a Bash call: pass through

    command = payload.get('tool_input', {}).get('command', '')
    allowlist = load_allowlist()
    allowed, reason = validate_command(command, allowlist)

    if allowed:
        sys.exit(0)
    else:
        output = {
            "hookSpecificOutput": {
                "hookEventName": "PreToolUse",
                "permissionDecision": "deny",
                "permissionDecisionReason": (
                    f"safecmd blocked this command: {reason}\n"
                    f"Command: {command!r}\n"
                    f"To allow this command, add it to .claude/safecmd_allowlist.json"
                )
            }
        }
        print(json.dumps(output))
        sys.exit(1)


if __name__ == '__main__':
    main()

## exhash PostToolUse Hook

After every Edit or Write tool call, captures the hash-addressed snapshot of the modified file and appends it to `.claude/edit_audit.jsonl`.

In [ ]:
#| default_exp hooks.exhash_hook

In [ ]:
#| export
from __future__ import annotations
import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

In [ ]:
#| export
def capture_lnhashview(file_path: str) -> str | None:
    """Run `lnhashview` on the file and return its output, or None if unavailable."""
    try:
        result = subprocess.run(
            ['lnhashview', file_path],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0:
            return result.stdout
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass
    return None

In [ ]:
#| export
def append_audit(audit_path: Path, record: dict) -> None:
    "Append a record to the JSONL audit log."
    audit_path.parent.mkdir(parents=True, exist_ok=True)
    with audit_path.open('a') as f:
        f.write(json.dumps(record) + '\n')

In [ ]:
#| export
def main():
    """Claude Code PostToolUse hook entry point. Reads JSON from stdin, logs edit audit."""
    try:
        payload = json.load(sys.stdin)
    except (json.JSONDecodeError, EOFError):
        sys.exit(0)

    tool_name = payload.get('tool_name', '')
    if tool_name not in ('Edit', 'Write'):
        sys.exit(0)

    tool_input = payload.get('tool_input', {})
    file_path = tool_input.get('file_path', '')

    if not file_path or not Path(file_path).exists():
        sys.exit(0)

    lnhash_snapshot = capture_lnhashview(file_path)

    record = {
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'tool': tool_name,
        'file': file_path,
        'lnhash_snapshot': lnhash_snapshot,
    }

    audit_path = Path.cwd() / '.claude' / 'edit_audit.jsonl'
    append_audit(audit_path, record)
    sys.exit(0)


if __name__ == '__main__':
    main()

## SessionStart Index Hook

On session start, extracts code signatures from the project using `codesigs` and builds a hybrid FTS5+vector search index with `litesearch`.

In [ ]:
#| default_exp hooks.index_hook

In [ ]:
#| export
from __future__ import annotations
import json
import sys
from pathlib import Path

In [ ]:
#| export
EXCLUDED_DIRS = {'.venv', 'venv', '.git', '__pycache__', 'node_modules', '.claude', 'dist', 'build'}
INDEXED_EXTENSIONS = {'.py', '.js', '.ts', '.jsx', '.tsx', '.rs', '.go', '.java'}

In [ ]:
#| export
def collect_source_files(root: Path) -> list[Path]:
    """Collect source files to index, excluding common non-source directories."""
    files = []
    for p in root.rglob('*'):
        if any(part in EXCLUDED_DIRS for part in p.parts):
            continue
        if p.suffix in INDEXED_EXTENSIONS and p.is_file():
            files.append(p)
    return files

In [ ]:
#| export
def build_index(root: Path, db_path: Path) -> int:
    """Build the code index. Returns number of items indexed.
    
    Uses codesigs to extract signatures, then litesearch to index them.
    Falls back to plain-text indexing if either library is unavailable.
    """
    files = collect_source_files(root)
    if not files:
        return 0

    docs = []
    try:
        from codesigs import file_sigs
        for f in files:
            try:
                sigs = file_sigs(str(f))
                for sig in sigs:
                    docs.append({
                        'text': f'{sig.get("signature", "")}\n{sig.get("docstring", "")}'.strip(),
                        'metadata': json.dumps({
                            'id': f'{f}::{sig.get("name", "")}',
                            'file': str(f),
                            'name': sig.get('name', ''),
                            'kind': sig.get('kind', 'function'),
                        }),
                    })
            except Exception:
                continue
    except ImportError:
        # Fallback: index raw file content
        for f in files:
            try:
                docs.append({
                    'text': f.read_text(errors='ignore')[:4000],
                    'metadata': json.dumps({'id': str(f), 'file': str(f)}),
                })
            except Exception:
                continue

    if not docs:
        return 0

    try:
        from litesearch import database
        from litesearch.utils import FastEncode
        db = database(str(db_path))
        store = db.get_store()
        encoder = FastEncode()
        rows = [
            {
                'content': d['text'],
                'embedding': encoder.encode_document(d['text']).tobytes(),
                'metadata': d['metadata'],
            }
            for d in docs if d['text'].strip()
        ]
        if rows:
            store.insert_all(rows)
        return len(rows)
    except ImportError:
        pass

    return 0

In [ ]:
#| export
def main():
    """Claude Code SessionStart hook entry point. Builds code search index silently."""
    try:
        # May or may not have payload on stdin for SessionStart
        payload_str = sys.stdin.read()
        payload = json.loads(payload_str) if payload_str.strip() else {}
    except Exception:
        payload = {}

    root = Path(payload.get('cwd', str(Path.cwd())))
    db_path = root / '.claude' / 'code_index.db'
    db_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        n = build_index(root, db_path)
        if n > 0:
            # Communicate back to Claude via stdout additional context
            output = {
                "hookSpecificOutput": {
                    "hookEventName": "SessionStart",
                    "additionalContext": (
                        f"Code index built: {n} signatures indexed into .claude/code_index.db\n"
                        "Use the /litesearch skill to query the index semantically."
                    )
                }
            }
            print(json.dumps(output))
    except Exception as e:
        pass  # Never block session start

    sys.exit(0)


if __name__ == '__main__':
    main()